In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

n_estimators_list = [100, 200, 300, 400, 500, 800]
contamination_list = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, "auto"]


def contamination_to_str(cont):
    if isinstance(cont, str):
        return cont
    return str(cont).replace(".", "_")


train_path = Path("../data_raw/ECG5000/ECG5000_TRAIN.txt")
test_path = Path("../data_raw/ECG5000/ECG5000_TEST.txt")
results_root = Path("../results/iForest_ECG")
results_root.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(train_path, header=None, sep=r"\s+")
test_df = pd.read_csv(test_path, header=None, sep=r"\s+")

df = pd.concat([train_df, test_df], ignore_index=True)

y_raw = df.iloc[:, 0].astype(int)
X = df.iloc[:, 1:].copy()

y_true = (y_raw != 1).astype(int)

row_mean = X.mean(axis=1)
row_std = X.std(axis=1).replace(0, 1e-8)

Xz = X.sub(row_mean, axis=0).div(row_std, axis=0)

X_normal = Xz[y_true == 0].copy()
X_anomaly = Xz[y_true == 1].copy()

X_train = X_normal.copy()

X_test_normal = X_normal.copy()
X_test_anomaly = X_anomaly.sample(n=min(150, len(X_anomaly)), random_state=42)

X_test = pd.concat([X_test_normal, X_test_anomaly], axis=0)
y_test = pd.Series(
    [0] * len(X_test_normal) + [1] * len(X_test_anomaly),
    index=X_test.index
)

test_mix_df = X_test.copy()
test_mix_df["y_true"] = y_test
test_mix_df = test_mix_df.sample(frac=1, random_state=42).reset_index(drop=True)

X_test = test_mix_df.drop(columns=["y_true"])
y_test = test_mix_df["y_true"].astype(int)

all_results = []

for contamination in contamination_list:
    contamination_dir = results_root / f"contamination_{contamination_to_str(contamination)}"
    contamination_dir.mkdir(parents=True, exist_ok=True)

    contamination_results = []

    for n_estimators in n_estimators_list:
        exp_name = f"n_estimators_{n_estimators}"
        exp_dir = contamination_dir / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)

        model = IsolationForest(
            n_estimators=n_estimators,
            max_samples="auto",
            contamination=contamination,
            max_features=1.0,
            bootstrap=False,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train)

        pred_raw = model.predict(X_test)
        decision_scores = model.decision_function(X_test)
        score_samples = model.score_samples(X_test)

        df_temp = X_test.copy()
        df_temp["y_true"] = y_test.values
        df_temp["iforest_pred_raw"] = pred_raw
        df_temp["iforest_decision"] = decision_scores
        df_temp["iforest_score"] = score_samples
        df_temp["is_anomaly"] = (df_temp["iforest_pred_raw"] == -1).astype(int)

        y_pred = df_temp["is_anomaly"]

        cm = confusion_matrix(y_test, y_pred)

        accuracy = accuracy_score(y_test, y_pred) * 100
        precision = precision_score(y_test, y_pred, zero_division=0) * 100
        recall = recall_score(y_test, y_pred, zero_division=0) * 100
        f1 = f1_score(y_test, y_pred, zero_division=0) * 100

        result_row = {
            "experiment": exp_name,
            "n_estimators": n_estimators,
            "contamination": contamination,
            "accuracy_%": round(accuracy, 2),
            "precision_%": round(precision, 2),
            "recall_%": round(recall, 2),
            "f1_%": round(f1, 2)
        }

        all_results.append(result_row)
        contamination_results.append(result_row)

        df_temp.to_csv(exp_dir / "predictions.csv", index=False)

        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(weight="bold")
                cell.set_facecolor("#dce6f1")
            else:
                cell.set_facecolor("#f9f9f9")

        plt.title("Výsledné metriky modelu Isolation Forest – ECG5000", fontsize=12, pad=20)
        plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title("Matica zámen (Confusion Matrix)")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > threshold else "white"
                plt.text(
                    j, i, cm[i, j],
                    ha="center", va="center",
                    color=color
                )

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        signal_cols = X_test.columns
        feature_only = df_temp.drop(columns=[
            "y_true", "iforest_pred_raw", "iforest_decision", "iforest_score", "is_anomaly"
        ])

        normal_idx = df_temp[df_temp["y_true"] == 0].index
        anomaly_idx = df_temp[df_temp["y_true"] == 1].index
        tp_idx = df_temp[(df_temp["y_true"] == 1) & (df_temp["is_anomaly"] == 1)].index
        fn_idx = df_temp[(df_temp["y_true"] == 1) & (df_temp["is_anomaly"] == 0)].index
        fp_idx = df_temp[(df_temp["y_true"] == 0) & (df_temp["is_anomaly"] == 1)].index

        if len(normal_idx) > 0:
            idx = normal_idx[0]
            plt.figure(figsize=(12, 4))
            plt.plot(signal_cols, feature_only.loc[idx].values)
            plt.title("Príklad normálneho EKG cyklu")
            plt.xlabel("Vzorka")
            plt.ylabel("Normalizovaná hodnota")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(exp_dir / "normal_signal.png", dpi=300, bbox_inches="tight")
            plt.close()

        if len(anomaly_idx) > 0:
            idx = anomaly_idx[0]
            plt.figure(figsize=(12, 4))
            plt.plot(signal_cols, feature_only.loc[idx].values)
            plt.title("Príklad anomálneho EKG cyklu")
            plt.xlabel("Vzorka")
            plt.ylabel("Normalizovaná hodnota")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(exp_dir / "anomaly_signal.png", dpi=300, bbox_inches="tight")
            plt.close()

        if len(tp_idx) > 0:
            idx = tp_idx[0]
            plt.figure(figsize=(12, 4))
            plt.plot(signal_cols, feature_only.loc[idx].values)
            plt.title("Správne detegovaná anomália (True Positive)")
            plt.xlabel("Vzorka")
            plt.ylabel("Normalizovaná hodnota")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(exp_dir / "true_positive.png", dpi=300, bbox_inches="tight")
            plt.close()

        if len(fn_idx) > 0:
            idx = fn_idx[0]
            plt.figure(figsize=(12, 4))
            plt.plot(signal_cols, feature_only.loc[idx].values)
            plt.title("Prehliadnutá anomália (False Negative)")
            plt.xlabel("Vzorka")
            plt.ylabel("Normalizovaná hodnota")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(exp_dir / "false_negative.png", dpi=300, bbox_inches="tight")
            plt.close()

        if len(fp_idx) > 0:
            idx = fp_idx[0]
            plt.figure(figsize=(12, 4))
            plt.plot(signal_cols, feature_only.loc[idx].values)
            plt.title("Falošne označený normálny cyklus (False Positive)")
            plt.xlabel("Vzorka")
            plt.ylabel("Normalizovaná hodnota")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(exp_dir / "false_positive.png", dpi=300, bbox_inches="tight")
            plt.close()

    contamination_results_df = pd.DataFrame(contamination_results)
    contamination_results_df = contamination_results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
    contamination_results_df.to_csv(contamination_dir / "summary_results.csv", index=False)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Done
